# PICO Research Question Nanopublication Creator (Multi-PICO Version)

Creates **multiple** PICO nanopublications from a single JSON configuration file.

**Template:** [Cochrane PICO Research Question](https://w3id.org/np/RA5e5XeXy_-aNK5giB7kBAEQslTLVydHeM4YYEzhmEE2w)

This template uses the Cochrane PICO ontology (`http://data.cochrane.org/ontologies/pico/`).

---

## Instructions

1. **Create a JSON file** with your PICO details (see template at bottom)
2. **Set the path** to your JSON file in Section 1
3. **Run All Cells** → Get multiple `.trig` files (one per PICO question)

---
# 📁 SECTION 1: INPUT FILE (EDIT THIS)
---

In [16]:
# Path to your PICO JSON file (with multiple nanopublications)
CONFIG_FILE = "/Users/annef/Documents/FAIR2Adapt/nanopub-notebooks/biodiversity/crete/crete_declaration_pico.json"
CONFIG_FILE = "/Users/annef/Documents/FAIR2Adapt/nanopub-notebooks/biodiversity/dimuri2022/dimuri2022_pico.json"

# Output directory for .trig files
OUTPUT_DIR = "../output/biodiversity"

---
# ⚙️ SECTION 2: SETUP
---

In [17]:
# Install dependencies (uncomment if needed)
# !pip install nanopub rdflib

In [18]:
import json
import re
import os
from rdflib import Graph, Dataset, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, XSD, FOAF
from datetime import datetime, timezone
from pathlib import Path

# Namespaces
NP = Namespace("http://www.nanopub.org/nschema#")
DCT = Namespace("http://purl.org/dc/terms/")
NT = Namespace("https://w3id.org/np/o/ntemplate/")
NPX = Namespace("http://purl.org/nanopub/x/")
PROV = Namespace("http://www.w3.org/ns/prov#")
ORCID = Namespace("https://orcid.org/")
PICO = Namespace("http://data.cochrane.org/ontologies/pico/")
SCIENCELIVE = Namespace("https://w3id.org/sciencelive/o/terms/")

# PICO template URI (Cochrane PICO ontology)
PICO_TEMPLATE = URIRef("https://w3id.org/np/RA5e5XeXy_-aNK5giB7kBAEQslTLVydHeM4YYEzhmEE2w")

# Template references
PROV_TEMPLATE = URIRef("https://w3id.org/np/RA7lSq6MuK_TIC6JMSHvLtee3lpLoZDOqLJCLXevnrPoU")
PUBINFO_TEMPLATE_1 = URIRef("https://w3id.org/np/RA0J4vUn_dekg-U1kK3AOEt02p9mT2WO03uGxLDec1jLw")
PUBINFO_TEMPLATE_2 = URIRef("https://w3id.org/np/RAukAcWHRDlkqxk7H2XNSegc1WnHI569INvNr-xdptDGI")
PUBINFO_TEMPLATE_3 = URIRef("https://w3id.org/np/RAoTD7udB2KtUuOuAe74tJi1t3VzK0DyWS7rYVAq1GRvw")

# Question type mapping (Science Live URIs)
QUESTION_TYPE_MAP = {
    "causation": SCIENCELIVE.CausationResearchQuestion,
    "descriptive": SCIENCELIVE.DescriptiveResearchQuestion,
    "effectiveness": SCIENCELIVE.EffectivenessResearchQuestions,
    "experience": SCIENCELIVE.ExperienceResearchQuestions,
    "prediction": SCIENCELIVE.PredictionResearchQuestions,
}

VALID_QUESTION_TYPES = list(QUESTION_TYPE_MAP.keys())

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✓ Setup complete")
print(f"  Template: {PICO_TEMPLATE}")

✓ Setup complete
  Template: https://w3id.org/np/RA5e5XeXy_-aNK5giB7kBAEQslTLVydHeM4YYEzhmEE2w


---
# 📖 SECTION 3: LOAD & VALIDATE
---

In [19]:
# Load PICO from JSON
print(f"Loading: {CONFIG_FILE}")

with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
    config = json.load(f)

# Extract metadata
AUTHOR_ORCID = config['metadata']['creator_orcid']
AUTHOR_NAME = config['metadata']['creator_name']
FILENAME_PREFIX = config.get('output', {}).get('filename_prefix', 'pico')

# Get source paper info if available
SOURCE_PAPER = config['metadata'].get('source_paper', {})

# Get all PICO questions
pico_list = config['nanopublications']

print(f"✓ Loaded {len(pico_list)} PICO questions")
print(f"  Author: {AUTHOR_NAME} ({AUTHOR_ORCID})")
if SOURCE_PAPER:
    print(f"  Source: {SOURCE_PAPER.get('title', 'N/A')}")
print()
for i, pico in enumerate(pico_list, 1):
    label = pico['label']
    print(f"  {i}. {label[:60]}..." if len(label) > 60 else f"  {i}. {label}")

Loading: /Users/annef/Documents/FAIR2Adapt/nanopub-notebooks/biodiversity/dimuri2022/dimuri2022_pico.json
✓ Loaded 1 PICO questions
  Author: Cristina Di Muri (0000-0003-4072-0662)
  Source: Trophic Ecology of Invasive Blue Crab in Mediterranean Coastal Ecosystems

  1. Trophic Ecology of Invasive Blue Crab in Mediterranean Coast...


In [20]:
# Validate all PICO entries
print("Validating...")

all_errors = []

# Check metadata
if not AUTHOR_ORCID or AUTHOR_ORCID == "0000-0000-0000-0000":
    all_errors.append("metadata.creator_orcid must be set to your real ORCID")
if not AUTHOR_NAME or AUTHOR_NAME == "Your Name":
    all_errors.append("metadata.creator_name must be set to your real name")

# Check each PICO
for i, pico in enumerate(pico_list):
    prefix = f"nanopublications[{i}]"
    if not pico.get('id'):
        all_errors.append(f"{prefix}.id is required")
    if not pico.get('label') or len(pico.get('label', '')) < 10:
        all_errors.append(f"{prefix}.label must be at least 10 characters")
    if not pico.get('population'):
        all_errors.append(f"{prefix}.population is required")
    if not pico.get('intervention'):
        all_errors.append(f"{prefix}.intervention is required")
    # comparison can be empty or "Not applicable"
    if not pico.get('outcome'):
        all_errors.append(f"{prefix}.outcome is required")
    if not pico.get('research_question'):
        all_errors.append(f"{prefix}.research_question is required")
    if pico.get('question_type') not in VALID_QUESTION_TYPES:
        all_errors.append(f"{prefix}.question_type must be one of: {VALID_QUESTION_TYPES}")

if all_errors:
    print("❌ Validation errors:")
    for e in all_errors:
        print(f"   - {e}")
    raise ValueError("Please fix the errors in your JSON file")
else:
    print("✓ All fields valid")

Validating...
✓ All fields valid


---
# 🔨 SECTION 4: BUILD NANOPUBLICATIONS
---

In [21]:
def make_slug(text):
    """Convert title to URI-safe slug."""
    slug = text.lower()
    slug = re.sub(r'[^a-z0-9\s-]', '', slug)
    slug = re.sub(r'[\s_]+', '-', slug)
    slug = re.sub(r'-+', '-', slug)
    return slug[:80].strip('-')


def create_pico_nanopub(pico_data, author_orcid, author_name):
    """
    Create a single PICO nanopublication.
    
    Args:
        pico_data: Dict with id, label, population, intervention, comparison, outcome, research_question, question_type
        author_orcid: ORCID of the author
        author_name: Name of the author
    
    Returns:
        Serialized TriG string
    """
    # Create namespaces
    TEMP_NP = Namespace("http://purl.org/nanopub/temp/np/")
    
    this_np = URIRef("http://purl.org/nanopub/temp/np/")
    head_graph = URIRef("http://purl.org/nanopub/temp/np/Head")
    assertion_graph = URIRef("http://purl.org/nanopub/temp/np/assertion")
    provenance_graph = URIRef("http://purl.org/nanopub/temp/np/provenance")
    pubinfo_graph = URIRef("http://purl.org/nanopub/temp/np/pubinfo")
    
    ds = Dataset()
    
    # Bind prefixes
    ds.bind("this", "http://purl.org/nanopub/temp/np/")
    ds.bind("sub", "http://purl.org/nanopub/temp/np/")
    ds.bind("np", NP)
    ds.bind("dct", DCT)
    ds.bind("nt", NT)
    ds.bind("npx", NPX)
    ds.bind("prov", PROV)
    ds.bind("orcid", ORCID)
    ds.bind("rdfs", RDFS)
    ds.bind("xsd", XSD)
    ds.bind("foaf", FOAF)
    ds.bind("pico", PICO)
    ds.bind("sciencelive", SCIENCELIVE)
    
    # === HEAD GRAPH ===
    head = ds.graph(head_graph)
    head.add((this_np, RDF.type, NP.Nanopublication))
    head.add((this_np, NP.hasAssertion, assertion_graph))
    head.add((this_np, NP.hasProvenance, provenance_graph))
    head.add((this_np, NP.hasPublicationInfo, pubinfo_graph))
    
    # === ASSERTION GRAPH ===
    assertion = ds.graph(assertion_graph)
    
    pico_slug = make_slug(pico_data['label'])
    pico_uri = TEMP_NP[pico_slug]
    
    # Create local resources for PICO components
    population_uri = TEMP_NP["population"]
    intervention_uri = TEMP_NP["interventionGroup"]
    comparator_uri = TEMP_NP["comparatorGroup"]
    outcome_uri = TEMP_NP["outcomeGroup"]
    
    # Main PICO resource - type it as BOTH pico:PICO AND the Science Live question type
    assertion.add((pico_uri, RDF.type, PICO.PICO))
    question_type_uri = QUESTION_TYPE_MAP.get(pico_data['question_type'])
    if question_type_uri:
        assertion.add((pico_uri, RDF.type, question_type_uri))
    
    # Label and description
    assertion.add((pico_uri, RDFS.label, Literal(pico_data['label'])))
    assertion.add((pico_uri, DCT.description, Literal(pico_data['research_question'])))
    
    # PICO components using Cochrane PICO predicates
    # Population
    assertion.add((pico_uri, PICO.population, population_uri))
    assertion.add((population_uri, DCT.description, Literal(pico_data['population'])))
    
    # Intervention
    assertion.add((pico_uri, PICO.interventionGroup, intervention_uri))
    assertion.add((intervention_uri, DCT.description, Literal(pico_data['intervention'])))
    
    # Comparator
    comparison = pico_data.get('comparison', 'Not applicable')
    assertion.add((pico_uri, PICO.comparatorGroup, comparator_uri))
    assertion.add((comparator_uri, DCT.description, Literal(comparison)))
    
    # Outcome
    assertion.add((pico_uri, PICO.outcomeGroup, outcome_uri))
    assertion.add((outcome_uri, DCT.description, Literal(pico_data['outcome'])))
    
    # === PROVENANCE GRAPH ===
    provenance = ds.graph(provenance_graph)
    author_uri = ORCID[author_orcid]
    provenance.add((assertion_graph, PROV.wasAttributedTo, author_uri))
    
    # === PUBINFO GRAPH ===
    pubinfo = ds.graph(pubinfo_graph)
    
    # Creator info
    pubinfo.add((author_uri, FOAF.name, Literal(author_name)))
    
    # Nanopub metadata
    timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S+00:00")
    pubinfo.add((this_np, DCT.created, Literal(timestamp, datatype=XSD.dateTime)))
    pubinfo.add((this_np, DCT.creator, author_uri))
    pubinfo.add((this_np, DCT.license, URIRef("https://creativecommons.org/licenses/by/4.0/")))
    pubinfo.add((this_np, NPX.wasCreatedAt, URIRef("https://platform.sciencelive4all.org/")))
    
    # CRITICAL: npx:introduces enables federated SPARQL queries to find this resource
    pubinfo.add((this_np, NPX.introduces, pico_uri))
    
    # Label (truncate if needed)
    label = f"PICO Research Question: {pico_data['label']}"
    if len(label) > 100:
        label = label[:97] + "..."
    pubinfo.add((this_np, RDFS.label, Literal(label)))
    
    # Template references
    pubinfo.add((this_np, NT.wasCreatedFromProvenanceTemplate, PROV_TEMPLATE))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_1))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_2))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_3))
    pubinfo.add((this_np, NT.wasCreatedFromTemplate, PICO_TEMPLATE))
    
    return ds.serialize(format="trig"), pico_uri

print("✓ Function defined")

✓ Function defined


In [7]:
# Generate all nanopublications
created_files = []

for i, pico in enumerate(pico_list, 1):
    print(f"\n[{i}/{len(pico_list)}] Creating: {pico['label'][:50]}...")
    
    # Create the nanopub
    trig_output, pico_uri = create_pico_nanopub(pico, AUTHOR_ORCID, AUTHOR_NAME)
    
    # Generate filename
    filename = f"{FILENAME_PREFIX}_{pico['id']}.trig"
    output_path = Path(OUTPUT_DIR) / filename
    
    # Save
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(trig_output)
    
    created_files.append(output_path)
    print(f"  ✓ Saved: {output_path}")
    print(f"    P: {pico['population'][:50]}...")
    print(f"    I: {pico['intervention'][:50]}...")
    print(f"    C: {pico.get('comparison', 'N/A')[:50]}...")
    print(f"    O: {pico['outcome'][:50]}...")

print(f"\n✓ Created {len(created_files)} PICO nanopublications")


[1/6] Creating: Effect of open science practices on research impac...
  ✓ Saved: ../output/biodiversity/crete_pico_pico_open_science_citation.trig
    P: Research outputs (publications, datasets, software...
    I: Open science practices: open access publishing, FA...
    C: Traditional closed/restricted access research outp...
    O: Research impact metrics: citation rates, data reus...

[2/6] Creating: Effect of FAIR data practices on evidence-based po...
  ✓ Saved: ../output/biodiversity/crete_pico_pico_fair_data_policy.trig
    P: Research datasets used in European environmental a...
    I: FAIR-compliant data management (Findable, Accessib...
    C: Non-FAIR or partially compliant data resources...
    O: Policy uptake indicators: citation in policy docum...

[3/6] Creating: Effect of temperature increase on antimicrobial re...
  ✓ Saved: ../output/biodiversity/crete_pico_pico_climate_amr.trig
    P: Bacterial isolates from clinical and environmental...
    I: Exposure to increas

---
# 📄 SECTION 5: SUMMARY
---

In [8]:
# Summary
print("=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Input:     {CONFIG_FILE}")
print(f"Output:    {OUTPUT_DIR}")
print(f"Author:    {AUTHOR_NAME} (orcid:{AUTHOR_ORCID})")
print(f"Template:  {PICO_TEMPLATE}")
print()
print(f"Created {len(created_files)} PICO nanopublications:")
print()
for i, (pico, path) in enumerate(zip(pico_list, created_files), 1):
    print(f"  {i}. {path.name}")
    print(f"     P: {pico['population'][:50]}...")
    print(f"     I: {pico['intervention'][:50]}...")
    print(f"     C: {pico.get('comparison', 'N/A')[:50]}...")
    print(f"     O: {pico['outcome'][:50]}...")
    print(f"     Type: {pico['question_type']}")
    print()

print("Next steps:")
print(f"  Sign all:    for f in {OUTPUT_DIR}*.trig; do nanopub sign \"$f\"; done")
print(f"  Publish all: for f in {OUTPUT_DIR}*.signed.trig; do nanopub publish \"$f\"; done")

SUMMARY
Input:     /Users/annef/Documents/FAIR2Adapt/nanopub-notebooks/biodiversity/crete/crete_declaration_pico.json
Output:    ../output/biodiversity
Author:    Anne Fouilloux (orcid:0000-0002-1784-2920)
Template:  https://w3id.org/np/RA5e5XeXy_-aNK5giB7kBAEQslTLVydHeM4YYEzhmEE2w

Created 6 PICO nanopublications:

  1. crete_pico_pico_open_science_citation.trig
     P: Research outputs (publications, datasets, software...
     I: Open science practices: open access publishing, FA...
     C: Traditional closed/restricted access research outp...
     O: Research impact metrics: citation rates, data reus...
     Type: effectiveness

  2. crete_pico_pico_fair_data_policy.trig
     P: Research datasets used in European environmental a...
     I: FAIR-compliant data management (Findable, Accessib...
     C: Non-FAIR or partially compliant data resources...
     O: Policy uptake indicators: citation in policy docum...
     Type: effectiveness

  3. crete_pico_pico_climate_amr.trig
     P: B

---
# 🚀 SECTION 6: SIGN & PUBLISH (OPTIONAL)
---

In [10]:
PUBLISH = True
USE_TEST_SERVER = False
PROFILE_PATH = "/Users/annef/Documents/ScienceLive/ai-profile/profile.yml" 

In [11]:
if PUBLISH:
    from nanopub import Nanopub, NanopubConf, load_profile
    
    if PROFILE_PATH:
        profile = load_profile(PROFILE_PATH)
    else:
        profile = load_profile()
    print(f"Loaded profile: {profile.name}")
    print("=" * 70)
    
    conf = NanopubConf(profile=profile, use_test_server=USE_TEST_SERVER)
    published_uris = []
    
    for i, trig_path in enumerate(created_files, 1):
        print(f"\n[{i}/{len(created_files)}] Processing {trig_path.name}...")
        
        np_obj = Nanopub(rdf=trig_path, conf=conf)
        
        np_obj.sign()
        print(f"  ✓ Signed")
        
        signed_path = trig_path.with_suffix('.signed.trig')
        np_obj.store(signed_path)
        print(f"  ✓ Saved: {signed_path.name}")
        
        np_obj.publish()
        print(f"  ✓ Published: {np_obj.source_uri}")
        published_uris.append(np_obj.source_uri)
    
    print("\n" + "=" * 70)
    print(f"✓ Published {len(published_uris)} nanopublications:")
    for uri in published_uris:
        print(f"  {uri}")
else:
    print("Publishing disabled. Set PUBLISH = True to enable.")

Loaded profile: claude-ai-agent

[1/6] Processing crete_pico_pico_open_science_citation.trig...
  ✓ Signed
  ✓ Saved: crete_pico_pico_open_science_citation.signed.trig
  ✓ Published: https://w3id.org/np/RAWnL-4tlqf_3h7cEAAHy-CcVrLKEWW8bxDMbutKAGaKw

[2/6] Processing crete_pico_pico_fair_data_policy.trig...
  ✓ Signed
  ✓ Saved: crete_pico_pico_fair_data_policy.signed.trig
  ✓ Published: https://w3id.org/np/RAMfgt7HScm-6gRVhUYAapIPhGTkcXr_3Uu3n_eM7EsTY

[3/6] Processing crete_pico_pico_climate_amr.trig...
  ✓ Signed
  ✓ Saved: crete_pico_pico_climate_amr.signed.trig
  ✓ Published: https://w3id.org/np/RAxHFsc7u5QDdKJuh1gz1Cuba-ixjwrMQSZE7ErpbId6k

[4/6] Processing crete_pico_pico_citizen_science_adoption.trig...
  ✓ Signed
  ✓ Saved: crete_pico_pico_citizen_science_adoption.signed.trig
  ✓ Published: https://w3id.org/np/RAFi2LFTyR2-fB55hmmo3tnSav538qdwNXlC-xU9ncxw0

[5/6] Processing crete_pico_pico_ri_collaboration_response.trig...
  ✓ Signed
  ✓ Saved: crete_pico_pico_ri_collaboration_r

---
# 📋 JSON TEMPLATE (Multi-PICO)

Create a JSON file with this structure:

```json
{
  "metadata": {
    "creator_orcid": "0000-0000-0000-0000",
    "creator_name": "Your Name",
    "source_paper": {
      "title": "Paper Title",
      "doi": "https://doi.org/10.xxxx/xxxxx"
    }
  },
  "nanopublications": [
    {
      "id": "pico_question_1",
      "label": "Short title for question 1",
      "population": "Who or what is being studied",
      "intervention": "What intervention or exposure",
      "comparison": "Comparison group (or 'Not applicable')",
      "outcome": "What outcomes are measured",
      "research_question": "Your full research question?",
      "question_type": "effectiveness"
    },
    {
      "id": "pico_question_2",
      "label": "Short title for question 2",
      "population": "...",
      "intervention": "...",
      "comparison": "...",
      "outcome": "...",
      "research_question": "...",
      "question_type": "causation"
    }
  ],
  "output": {
    "filename_prefix": "my_project_pico"
  }
}
```

**Question types:** `causation`, `descriptive`, `effectiveness`, `experience`, `prediction`

**Note:** The `evaluation` type is NOT available in the Cochrane PICO template.

---

## PICO vs PCC

| Component | PICO | PCC |
|-----------|------|-----|
| P | Population | Population |
| I | Intervention | - |
| C | Comparison | Concept |
| O | Outcome | Context |

Use **PICO** for:
- Clinical trials
- Intervention studies
- Effectiveness research
- Causation studies
- Comparative studies

Use **PCC** for:
- Scoping reviews
- Data papers (describing/documenting data)
- Qualitative research
- Descriptive studies
- Mapping/charting exercises

---